<a href="https://colab.research.google.com/github/detektor777/colab_list_video/blob/main/enhance_black.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://drive.google.com/drive

In [ ]:
#@title ##**Install** { display-mode: "form" }
%%capture
!wget https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz
!tar -xf ffmpeg-master-latest-linux64-gpl.tar.xz
!mv ffmpeg-master-latest-linux64-gpl/bin/ffmpeg /usr/local/bin/
!chmod +x /usr/local/bin/ffmpeg

In [ ]:
#@title ##**Select Video File** { display-mode: "form" }
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
from google.colab import drive

upload_option = "Load from Google Drive Root"  #@param ["Upload from PC", "Load from Google Drive Root", "Load from Google Drive"]

file_name = None
last_selected_button = None

def reset_button_colors(buttons):
    for btn in buttons:
        btn.style.button_color = None

if upload_option == "Upload from PC":
    print("Please upload a video file.")
    root_dir = '/content/'
    uploaded = files.upload()
    if uploaded:
        file_name = list(uploaded.keys())[0]
    else:
        print("No file uploaded.")
        file_name = None

elif upload_option == "Load from Google Drive Root":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for f in os.listdir(root_dir):
        if os.path.isfile(os.path.join(root_dir, f)) and os.path.splitext(f)[1].lower() in video_extensions:
            files_list.append(f)

    if not files_list:
        print("No video files found in Google Drive root.")
        file_name = None
    else:
        print("Select a video file from Google Drive root:")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

elif upload_option == "Load from Google Drive":
    drive.mount('/content/drive')
    root_dir = '/content/drive/MyDrive/'

    video_extensions = ['.mp4', '.mkv', '.avi', '.mov']
    files_list = []

    for dirpath, _, filenames in os.walk(root_dir):
        for f in filenames:
            if os.path.splitext(f)[1].lower() in video_extensions:
                relative_path = os.path.relpath(os.path.join(dirpath, f), root_dir)
                files_list.append(relative_path)

    if not files_list:
        print("No video files found in Google Drive or its subfolders.")
        file_name = None
    else:
        print("Select a video file from Google Drive (including subfolders):")

        output = widgets.Output()
        buttons = []

        def on_button_clicked(b):
            global file_name, last_selected_button
            with output:
                clear_output()
                reset_button_colors(buttons)
                selected_file = b.description
                file_name = os.path.join(root_dir, selected_file)

                if file_name and os.path.exists(file_name):
                    b.style.button_color = 'green'
                else:
                    b.style.button_color = 'red'

                last_selected_button = b
                print(f"Selected file: {file_name if file_name else 'None'}")

        for file in files_list:
            button = widgets.Button(description=file, layout=widgets.Layout(width='500px', overflow='hidden', text_overflow='ellipsis'))
            button.on_click(on_button_clicked)
            buttons.append(button)

        display(widgets.VBox(buttons), output)

if file_name:
    print(f"Video file path set to: {file_name}")
else:
    print("Video file path not set. Please select a file.")

In [ ]:
#@title ##**Files Config** { display-mode: "form" }
import os
from google.colab import files
import shutil
from google.colab import drive
output_folder = "google_drive" #@param ["google_drive","root"]

upload_folder = 'upload'
result_folder = 'results'

if output_folder == "google_drive":
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    root_folder = '/content/drive/MyDrive/';
    real_output_folder = '/content/drive/MyDrive/real_output'
    real_input_folder = "/content/drive/MyDrive/real_input"
elif output_folder == "root":
    root_folder = '/content/';
    real_output_folder = '/content/real_output'
    real_input_folder = "/content/real_input"

if not os.path.exists(real_output_folder):
    os.makedirs(real_output_folder)

if not os.path.exists(real_input_folder):
    os.makedirs(real_input_folder)

#clear folders
clear_input_folder = False #@param {type:"boolean"}
up_to_frame = "" #@param {type:"string"}
from_frame = "" #@param {type:"string"}

def clean_folder(folder_path, up_to=None, from_frame=None):
    print(f"\nCurrent parameters:")
    print(f"Delete frames up to: {up_to if up_to else 'not specified'}")
    print(f"Delete frames after: {from_frame if from_frame else 'not specified'}")

    if not os.path.isdir(folder_path):
        print(f"\nFolder {folder_path} does not exist!")
        print("Creating a new folder...")
        os.makedirs(folder_path)
        return

    if not up_to and not from_frame:
        print("\nNo parameters specified - deleting all folder content...")
        shutil.rmtree(folder_path)
        os.makedirs(folder_path)
        print(f"Folder {folder_path} cleared and recreated")
        return

    print("\nStarting file processing...")
    files = os.listdir(folder_path)
    jpg_files = [f for f in files if f.endswith('.jpg')]

    if not jpg_files:
        print("No JPG files to process in the folder")
        return

    deleted_count = 0
    processed_count = 0

    for filename in jpg_files:
        try:
            frame_number = int(filename.split('.')[0])
            should_delete = False

            if up_to and from_frame:
                if frame_number < int(up_to) or frame_number > int(from_frame):
                    should_delete = True
            elif up_to:
                if frame_number < int(up_to):
                    should_delete = True
            elif from_frame:
                if frame_number > int(from_frame):
                    should_delete = True

            if should_delete:
                file_path = os.path.join(folder_path, filename)
                os.remove(file_path)
                deleted_count += 1
                if deleted_count <= 5:
                    print(f'File deleted: {filename}')
                elif deleted_count == 6:
                    print('...')
            else:
                processed_count += 1

        except ValueError:
            print(f'Skipped file with invalid name: {filename}')

    print(f'\nProcessing complete:')
    print(f'Total files: {len(jpg_files)}')
    print(f'Files deleted: {deleted_count}')
    print(f'Files retained: {processed_count}')

if clear_input_folder:
    up_to_frame = up_to_frame if up_to_frame != "0" else None
    from_frame = from_frame if from_frame != "0" else None
    clean_folder(real_input_folder, up_to_frame, from_frame)

clear_output_folder = False #@param {type:"boolean"}

if clear_output_folder:
    if os.path.isdir(real_output_folder):
        shutil.rmtree(real_output_folder)
    os.makedirs(real_output_folder)

In [ ]:
#@title ##**Video Config & Preview** { display-mode: "form" }
from google.colab import files
import cv2
import numpy as np
import PIL.Image
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from ipywidgets import interactive
import base64
import io

if 'file_name' not in globals():
    print("No video file selected. Please run the video selection cell first.")
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

cap = cv2.VideoCapture(file_name)
if not cap.isOpened():
    raise ValueError(f"Failed to open video from {file_name}. Check the path.")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

display_width = 500
display_height = int(height * (display_width / width))

current_frame = None
current_frame_num = 0

def enhance_blacks(image, black_level=0.5, threshold=0.1, transition=0.05, curve_strength=0.5):
    image_float = image.astype(np.float32) / 255.0
    corrected = np.copy(image_float)
    mask = image_float < threshold
    linear_factor = (1 - black_level)
    nonlinear_factor = np.power(image_float[mask] / threshold, black_level * curve_strength)
    corrected[mask] = image_float[mask] * (
        (1 - curve_strength) * linear_factor + curve_strength * nonlinear_factor
    )
    transition_start = threshold - transition
    transition_end = threshold
    transition_mask = (image_float >= transition_start) & (image_float < transition_end)
    # Create a broadcastable mask for easier indexing
    broadcastable_transition_mask = np.expand_dims(transition_mask.any(axis=2), axis=2)

    # Check if there's anything to do in the transition zone
    if np.any(transition_mask):
        # Process each channel separately for the transition
        for channel in range(3):
            channel_mask = transition_mask[:, :, channel]
            if not np.any(channel_mask): continue

            alpha = (image_float[channel_mask, channel] - transition_start) / transition

            # Calculate the darkened value as if it were below the threshold
            darkened_value = image_float[channel_mask, channel] * (
                (1 - curve_strength) * linear_factor +
                curve_strength * np.power(image_float[channel_mask, channel] / threshold, black_level * curve_strength)
            )

            # Get the original value
            original_value = image_float[channel_mask, channel]

            # Blend between the darkened value and the original value
            corrected[channel_mask, channel] = (1 - alpha) * darkened_value + alpha * original_value

    corrected = np.clip(corrected, 0, 1) * 255
    return corrected.astype(np.uint8)

def create_combined_image(frame, black_level, threshold, transition, curve_strength, blend_factor=1.0):
    result_bgr = enhance_blacks(frame, black_level, threshold, transition, curve_strength)
    result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)

    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32)
    final_rgb = (1 - blend_factor) * frame_rgb + blend_factor * result_rgb

    image_original = PIL.Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image_enhanced = PIL.Image.fromarray(final_rgb.clip(0, 255).astype(np.uint8))

    max_width = 500
    width_original, height_original = image_original.size
    width_enhanced, height_enhanced = image_enhanced.size

    if width_original > max_width:
        new_height = int(height_original * max_width / width_original)
        image_original = image_original.resize((max_width, new_height))
    if width_enhanced > max_width:
        new_height = int(height_enhanced * max_width / width_enhanced)
        image_enhanced = image_enhanced.resize((max_width, new_height))

    combined_image = PIL.Image.fromarray(np.hstack((np.array(image_original), np.array(image_enhanced))))
    return combined_image

def update_frame(frame_num):
    global current_frame, current_frame_num
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_num)
    ret, frame = cap.read()
    if ret:
        current_frame = frame
        current_frame_num = frame_num
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_resized = cv2.resize(frame_rgb, (display_width, display_height))
        frame_pil = PIL.Image.fromarray(frame_resized)

        buffered = io.BytesIO()
        frame_pil.save(buffered, format="JPEG")
        img_str = base64.b64encode(buffered.getvalue()).decode()

        with video_output:
            clear_output(wait=True)
            display(HTML(f'<img src="data:image/jpeg;base64,{img_str}" />'))
            display(HTML(f'<p>Frame: {frame_num}/{frame_count-1}</p>'))

def on_test_button_clicked(b):
    global current_frame
    if current_frame is None:
        with result_output:
            clear_output(wait=True)
            print("No frame loaded. Please select a frame first.")
        return
    process_frame(current_frame)

def process_frame(frame):
    black_level = slider_black_level.value
    threshold = slider_threshold.value
    transition = slider_transition.value
    curve_strength = slider_curve_strength.value
    blend_factor = slider_blend.value

    combined_image = create_combined_image(frame, black_level, threshold, transition, curve_strength, blend_factor)

    with result_output:
        clear_output(wait=True)
        display(combined_image)
        display(HTML(f"<p>Applied settings - Black Level: {black_level:.2f}, Threshold: {threshold:.2f}, Transition: {transition:.2f}, Curve: {curve_strength:.2f}, Blend: {blend_factor:.2f}</p>"))

def on_slider_change(change):
    if current_frame is not None:
        process_frame(current_frame)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=frame_count-1,
    step=1,
    description='Frame:',
    continuous_update=False,
    orientation='horizontal',
    layout=widgets.Layout(width='500px')
)

prev_button = widgets.Button(description='Previous', icon='backward')
next_button = widgets.Button(description='Next', icon='forward')
test_button = widgets.Button(description='Test Frame', button_style='success')

slider_black_level = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.1, description='Black Level')
slider_threshold = widgets.FloatSlider(value=0.2, min=0.0, max=1.0, step=0.05, description='Threshold')
slider_transition = widgets.FloatSlider(value=0.1, min=0.0, max=0.2, step=0.01, description='Transition')
slider_curve_strength = widgets.FloatSlider(value=0.9, min=0.0, max=1.0, step=0.1, description='Curve')
slider_blend = widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.01, description='Blend Factor')

def on_prev_button_clicked(b):
    new_frame = max(0, frame_slider.value - 1)
    frame_slider.value = new_frame

def on_next_button_clicked(b):
    new_frame = min(frame_count-1, frame_slider.value + 1)
    frame_slider.value = new_frame

frame_slider.observe(lambda change: update_frame(change['new']), names='value')
prev_button.on_click(on_prev_button_clicked)
next_button.on_click(on_next_button_clicked)
test_button.on_click(on_test_button_clicked)

slider_black_level.observe(on_slider_change, names='value')
slider_threshold.observe(on_slider_change, names='value')
slider_transition.observe(on_slider_change, names='value')
slider_curve_strength.observe(on_slider_change, names='value')
slider_blend.observe(on_slider_change, names='value')

video_controls = widgets.HBox([prev_button, frame_slider, next_button])
video_output = widgets.Output()
result_output = widgets.Output()

display(widgets.HTML("<h3>Video Player</h3>"))
display(video_controls)
display(video_output)
display(widgets.HTML("<h3>Enhance Blacks</h3>"))
display(widgets.VBox([slider_black_level, slider_threshold, slider_transition, slider_curve_strength, slider_blend, test_button]))
display(result_output)

update_frame(0)

In [ ]:
#@title ##**Run sequence (Video to Frames)** { display-mode: "form" }
import cv2
import imageio
import os
import tqdm
import subprocess
import numpy as np
import time

library = "ffmpeg" #@param ["ffmpeg"]
delay = "0" #@param [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
decoding_mode = "software" #@param ["software", "hardware"]
jpg_quality = 90 #@param {type:"slider", min:0, max:100, step:1}

if library == "ffmpeg":
    !pip install ffmpeg-python
    import ffmpeg
    path = root_dir
    full_path = os.path.join(path, file_name)

    if not os.path.exists(full_path):
        raise FileNotFoundError(f"Video file not found: {full_path}")

    if not os.path.exists(real_input_folder):
        os.makedirs(real_input_folder)

    cuda_supported = False
    if decoding_mode == "hardware":
        try:
            check_cmd = 'ffmpeg -decoders'
            decoders = subprocess.check_output(check_cmd, shell=True, stderr=subprocess.STDOUT).decode('utf-8')
            if 'cuda' in decoders or 'nvdec' in decoders:
                cuda_supported = True
                print("CUDA/NVDEC supported, using hardware decoding.")
            else:
                print("CUDA/NVDEC not supported, switching to software decoding.")
                decoding_mode = "software"
        except subprocess.CalledProcessError:
            print("Error checking CUDA/NVDEC, switching to software decoding.")
            decoding_mode = "software"

    probe = ffmpeg.probe(full_path)
    video_info = next(stream for stream in probe['streams'] if stream['codec_type'] == 'video')
    fps = video_info['r_frame_rate']
    duration = float(video_info['duration'])
    frame_count = int(video_info['nb_frames'])

    print("FPS:", fps)
    print("Duration:", duration)
    print("Frames:", frame_count)
    print(f"JPEG Quality: {jpg_quality}")

    pbar_ffmpeg = tqdm.tqdm(total=frame_count, ncols=100, position=0, leave=True)
    process = (
        ffmpeg
        .input(full_path)
        .output('pipe:', format='rawvideo', pix_fmt='rgb24')
        .run_async(pipe_stdout=True)
        if decoding_mode == "software" else
        ffmpeg
        .input(full_path, **{'hwaccel': 'cuda'})
        .output('pipe:', format='rawvideo', pix_fmt='rgb24')
        .run_async(pipe_stdout=True)
    )

    for i in range(frame_count):
        try:
            raw_video = process.stdout.read(video_info['width'] * video_info['height'] * 3)
            if not raw_video:
                print(f"Reached end of video at frame {i}.")
                break
            frame = np.frombuffer(raw_video, dtype='uint8').reshape((video_info['height'], video_info['width'], 3))
            frame_path = f"{real_input_folder}/{i:09d}.jpg"
            if os.path.isfile(frame_path):
                pbar_ffmpeg.update(1)
                continue
            imageio.imwrite(frame_path, frame, quality=jpg_quality)
        except Exception as e:
            print(f"Error writing frame {i}: {str(e)}. Skipping...")
            continue
        pbar_ffmpeg.update(1)
        time.sleep(float(delay))

    pbar_ffmpeg.close()
    process.wait()

import os

def check_frames():
    frame_dir = real_input_folder
    frames = [int(f.split('.')[0]) for f in os.listdir(frame_dir) if f.endswith('.jpg')]
    if not frames:
        print("No frames found in folder.")
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print(f"Minimum frame: {min_frame}")
    print(f"Maximum frame: {max_frame}")

    missing_frames = []
    for i in range(min_frame, max_frame + 1):
        if i not in frames:
            missing_frames.append(i)

    if missing_frames:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

attempts = 0
max_attempts = 10

while attempts < max_attempts:
    try:
        check_frames()
        break
    except Exception as e:
        attempts += 1
        print(f"Attempt {attempts} failed: {str(e)}")
        if attempts == max_attempts:
            print("Maximum attempts reached. Check failed.")
        else:
            print("Retrying...")

In [ ]:
#@title ##**Run Enhance Blacks** { display-mode: "form" }
import shutil
from tqdm import tqdm
import os
import re
import cv2
import numpy as np
import glob

def check_frames(frame_dir):
    if not os.path.exists(frame_dir) or not os.listdir(frame_dir):
        print(f"Directory {frame_dir} is empty or does not exist.")
        return
    frames = [int(f.split('.')[0]) for f in os.listdir(frame_dir) if f.endswith('.jpg')]
    if not frames:
        print(f"No jpg frames found in {frame_dir}")
        return
    min_frame = min(frames)
    max_frame = max(frames)
    print(f"Min frame in {os.path.basename(frame_dir)}: {min_frame}, Max frame: {max_frame}")

    missing_frames = []
    for i in range(min_frame, max_frame + 1):
        if i not in frames:
            missing_frames.append(i)

    if missing_frames:
        print(f"Missing frames: {missing_frames}")
    else:
        print("All frames present")

print("--- Checking input folder before run ---")
check_frames(real_input_folder)
print("--- Checking output folder before run ---")
check_frames(real_output_folder)

def process_frame_for_run(img):
    # Use the parameter values from the interactive sliders
    black_level = slider_black_level.value
    threshold = slider_threshold.value
    transition = slider_transition.value
    curve_strength = slider_curve_strength.value
    blend_factor = slider_blend.value

    # Apply the enhancement
    enhanced_bgr = enhance_blacks(img, black_level, threshold, transition, curve_strength)

    # Blend with the original image
    if blend_factor < 1.0:
        final_bgr = cv2.addWeighted(enhanced_bgr, blend_factor, img, 1 - blend_factor, 0)
    else:
        final_bgr = enhanced_bgr

    return final_bgr.clip(0, 255).astype(np.uint8)


file_list = sorted(os.listdir(real_input_folder))
frames = [int(f.split('.')[0]) for f in file_list if f.endswith('.jpg')]
min_frame = min(frames)

real_files = os.listdir(real_output_folder)
start_frame = 0
if real_files:
    real_frames = [int(f.split('.')[0]) for f in real_files if f.endswith('.jpg')]
    if real_frames:
      start_frame = max(real_frames) + 1

max_frame = frames[-1]
files_to_copy = [f"{frame:09d}.jpg" for frame in range(start_frame, max_frame+1) if f"{frame:09d}.jpg" in file_list]

total_files = len(files_to_copy)
batch_size = 10
num_iterations = (total_files + batch_size - 1) // batch_size

print(f"\nStarting processing from frame: {start_frame}")
print(f"Total files to process: {total_files}")
print(f"Iterations: {num_iterations}")

upload_folder = '/content/upload'
result_folder = '/content/results'

if not os.path.isdir(upload_folder): os.makedirs(upload_folder)
if not os.path.isdir(result_folder): os.makedirs(result_folder)

with tqdm(total=num_iterations, desc="Processing Batches") as pbar:
    for i in range(0, total_files, batch_size):
        # 1. Clear temp folders
        shutil.rmtree(upload_folder)
        os.makedirs(upload_folder)
        shutil.rmtree(result_folder)
        os.makedirs(result_folder)

        # 2. Copy batch to upload folder
        batch_files = files_to_copy[i:i+batch_size]
        for file_basename in batch_files:
            shutil.copy(os.path.join(real_input_folder, file_basename), upload_folder)

        # 3. Process files from upload to results
        for img_path in glob.glob(os.path.join(upload_folder, "*.jpg")):
            img = cv2.imread(img_path)
            if img is None:
                print(f"Failed to load image: {img_path}")
                continue

            output = process_frame_for_run(img)

            imgname = os.path.basename(img_path)
            save_path = os.path.join(result_folder, imgname)
            cv2.imwrite(save_path, output, [cv2.IMWRITE_JPEG_QUALITY, jpg_quality])

        # 4. Copy results to final destination
        for item in os.listdir(result_folder):
            s = os.path.join(result_folder, item)
            d = os.path.join(real_output_folder, item)
            shutil.copy2(s, d)

        pbar.update(1)

print("\n--- Checking output folder after run ---")
check_frames(real_output_folder)

In [ ]:
#@title ##**Create video** { display-mode: "form" }
import cv2
import os
import subprocess
import time
from tqdm.notebook import tqdm
import torch
import gc

output_file_folder = "google_drive" #@param ["google_drive", "root"]
encoding_mode = "hardware" #@param ["software", "hardware"]
enhancement_blend_factor = 100 #@param {type:"slider", min:0, max:100, step:1}
bitrate_coefficient = 1 #@param {type:"slider", min:0.5, max:4, step:0.5}

ffmpeg_path = "/usr/local/bin/ffmpeg" # Using the ffmpeg installed in the first cell

nvenc_supported = False
effective_encoding_mode = encoding_mode
if encoding_mode == "hardware":
    try:
        encoders_output = subprocess.check_output([ffmpeg_path, "-encoders"], stderr=subprocess.STDOUT).decode('utf-8')
        if 'h264_nvenc' in encoders_output:
            nvenc_supported = True
            print("NVENC is supported.")
        else:
            print("NVENC not found. Switching to software encoding.")
            effective_encoding_mode = "software"
    except (subprocess.CalledProcessError, FileNotFoundError) as e:
        print(f"Could not check for NVENC support: {e}. Switching to software encoding.")
        effective_encoding_mode = "software"
else:
    print("Software encoding selected.")
    effective_encoding_mode = "software"

if output_file_folder == "google_drive":
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

gc.collect()

print(f"output_file_folder: {output_file_folder}")
print(f"Requested encoding mode: {encoding_mode}")
print(f"Effective encoding mode: {effective_encoding_mode}")

if 'file_name' in locals() and os.path.exists(file_name):
    base_file_name = os.path.basename(file_name)
else:
    raise ValueError("file_name is not defined or the file does not exist")

save_path = '/content/drive/MyDrive/' if output_file_folder == "google_drive" else '/content/'

output_file_name = base_file_name.rsplit('.', 1)[0] + f'_enhance_black_{enhancement_blend_factor}_{effective_encoding_mode}.mp4'
output_file = os.path.join(save_path, output_file_name)
temp_video = "/content/temp_video.mp4"

cap = cv2.VideoCapture(file_name)
fps_of_video = cap.get(cv2.CAP_PROP_FPS)
cap.release()

enhanced_img_files = sorted([os.path.join(real_output_folder, img) for img in os.listdir(real_output_folder) if img.lower().endswith('.jpg')])

if enhancement_blend_factor < 100:
    original_img_files = sorted([os.path.join(real_input_folder, img) for img in os.listdir(real_input_folder) if img.lower().endswith('.jpg')])
    if len(enhanced_img_files) != len(original_img_files):
        raise ValueError("Number of enhanced and original frames does not match")

if not enhanced_img_files:
    raise ValueError("No images found in the enhanced frames folder")

first_frame = cv2.imread(enhanced_img_files[0])
height, width, _ = first_frame.shape
del first_frame

def get_video_bitrate(file_path):
    try:
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'stream=bit_rate', '-of', 'default=noprint_wrappers=1:nokey=1', file_path]
        result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return int(result.stdout.strip())
    except (FileNotFoundError, ValueError):
        print("Could not determine bitrate. Using default.")
        return 7500000

original_bitrate = get_video_bitrate(file_name)
target_bitrate = int(original_bitrate * bitrate_coefficient)
bitrate_str = f'{target_bitrate // 1000}k'

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_video, fourcc, fps_of_video, (width, height))

if enhancement_blend_factor == 100:
    for img_file in tqdm(enhanced_img_files, desc="Writing frames"):
        frame = cv2.imread(img_file)
        out.write(frame)
else:
    alpha = enhancement_blend_factor / 100.0
    beta = 1.0 - alpha
    for enhanced_img, original_img in tqdm(zip(enhanced_img_files, original_img_files), total=len(enhanced_img_files), desc="Blending and writing frames"):
        enhanced_frame = cv2.imread(enhanced_img)
        original_frame = cv2.imread(original_img)
        # Ensure original frame is resized to match enhanced frame dimensions if they differ
        if original_frame.shape[:2] != (height, width):
            original_frame = cv2.resize(original_frame, (width, height))
        blended_frame = cv2.addWeighted(enhanced_frame, alpha, original_frame, beta, 0)
        out.write(blended_frame)

out.release()
gc.collect()

print("Re-encoding video...")
temp_converted = "/content/temp_converted.mp4"

if effective_encoding_mode == "hardware":
    max_bitrate = f'{int(target_bitrate * 1.5) // 1000}k'
    bufsize = f'{int(target_bitrate * 2) // 1000}k'
    cmd = [
        ffmpeg_path, '-i', temp_video, '-c:v', 'h264_nvenc',
        '-b:v', bitrate_str, '-maxrate', max_bitrate, '-bufsize', bufsize,
        '-preset', 'p7', '-y', temp_converted
    ]
else:
    cmd = [ffmpeg_path, '-i', temp_video, '-c:v', 'libx264', '-b:v', bitrate_str, '-y', temp_converted]

result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg re-encoding failed: {result.stderr}")
    raise RuntimeError("Conversion to H.264 failed")
else:
    print("Re-encoding successful.")

print("Muxing audio and subtitles...")
cmd = [ffmpeg_path, '-i', temp_converted, '-i', file_name, '-map', '0:v', '-map', '1:a?', '-map', '1:s?', '-c', 'copy', '-y', output_file]
result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
if result.returncode != 0:
    print(f"FFmpeg audio muxing failed: {result.stderr}")
    raise RuntimeError("Audio muxing failed")

if os.path.exists(output_file):
    if os.path.exists(temp_video): os.remove(temp_video)
    if os.path.exists(temp_converted): os.remove(temp_converted)
    print("Video created successfully")
    print(f"Video saved at: {output_file}")
else:
    print("Failed to create video")
    print(f"FFmpeg error output: {result.stderr}")

In [ ]:
#@title ##**Compare videos (optional)** { display-mode: "form" }
from IPython.display import display, HTML
import os
import base64

original_video_path = file_name
processed_video_path = output_file

if not os.path.exists(original_video_path):
    raise ValueError(f"Original video not found at path: {original_video_path}")
if not os.path.exists(processed_video_path):
    raise ValueError(f"Processed video not found at path: {processed_video_path}")

original_size = os.path.getsize(original_video_path) / (1024 * 1024)
processed_size = os.path.getsize(processed_video_path) / (1024 * 1024)
print(f"Original video size: {original_size:.2f} MB")
print(f"Processed video size: {processed_size:.2f} MB")

def video_to_base64(video_path):
    with open(video_path, "rb") as video_file:
        return base64.b64encode(video_file.read()).decode('utf-8')

original_base64 = video_to_base64(original_video_path)
processed_base64 = video_to_base64(processed_video_path)

html_code = f"""
<div style="display: flex; justify-content: center; flex-direction: column; align-items: center;">
    <div style="display: flex; justify-content: center;">
        <div style="margin-right: 10px;">
            <video id="originalVideo" width="400" controls preload="auto">
                <source src="data:video/mp4;base64,{original_base64}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            <p>Original Video</p>
        </div>
        <div>
            <video id="processedVideo" width="400" controls preload="auto">
                <source src="data:video/mp4;base64,{processed_base64}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            <p>Processed Video</p>
        </div>
    </div>
    <button id="playPauseBtn" style="margin-top: 10px; padding: 10px 20px; font-size: 16px;">Play</button>
</div>
<script>
(function() {{
    var originalVideo = document.getElementById("originalVideo");
    var processedVideo = document.getElementById("processedVideo");
    var playPauseBtn = document.getElementById("playPauseBtn");
    var isPlaying = false;

    function playBoth() {{
        Promise.all([originalVideo.play(), processedVideo.play()])
            .then(() => {{ isPlaying = true; playPauseBtn.textContent = "Pause"; }})
            .catch(error => console.log("Error playing videos:", error));
    }}

    function pauseBoth() {{
        originalVideo.pause();
        processedVideo.pause();
        isPlaying = false;
        playPauseBtn.textContent = "Play";
    }}

    playPauseBtn.addEventListener("click", () => {{ isPlaying ? pauseBoth() : playBoth(); }});

    function sync(master, slave) {{
        if (Math.abs(master.currentTime - slave.currentTime) > 0.1) {
            slave.currentTime = master.currentTime;
        }
    }

    originalVideo.addEventListener("play", () => {{ if (processedVideo.paused) processedVideo.play(); isPlaying = true; playPauseBtn.textContent = "Pause"; }});
    processedVideo.addEventListener("play", () => {{ if (originalVideo.paused) originalVideo.play(); isPlaying = true; playPauseBtn.textContent = "Pause"; }});
    originalVideo.addEventListener("pause", () => {{ if (!processedVideo.paused) processedVideo.pause(); isPlaying = false; playPauseBtn.textContent = "Play"; }});
    processedVideo.addEventListener("pause", () => {{ if (!originalVideo.paused) originalVideo.pause(); isPlaying = false; playPauseBtn.textContent = "Play"; }});
    originalVideo.addEventListener("timeupdate", () => sync(originalVideo, processedVideo));
    processedVideo.addEventListener("timeupdate", () => sync(processedVideo, originalVideo));

}})();
</script>
"""

display(HTML(html_code))

In [ ]:
#@title ##**Download video** { display-mode: "form" }
from google.colab import files
import os

if 'output_file' not in locals() or not os.path.exists(output_file):
    print("Video not found")
else:
    files.download(output_file)